# Challenge 05 -- Observability, Traceability, Monitoring, and Evaluation

Use this notebook with [Student/Challenge-05.md](../Challenge-05.md).

You will implement observability step by step, based on the flow in the referenced demo repository:
- tracing and traceability
- monitoring signals
- continuous evaluation
- batch evaluation


## How This Notebook Works

- Some cells are fully implemented so you can run quickly.
- Some cells include `___` blanks or TODO blocks for you to complete.
- You must run the observability workflow on at least one previously deployed agent from earlier challenges.
- Do not skip the reflection section at the end; that is where observability insights become action items.

Difficulty target: intermediate (not too easy, not too hard).

In [ ]:
%pip install azure-ai-projects azure-identity azure-ai-evaluation opentelemetry-api opentelemetry-sdk python-dotenv pandas --quiet


In [ ]:
import json
import os
import time

import pandas as pd
from dotenv import load_dotenv

load_dotenv()

PROJECT_ENDPOINT = os.getenv("AZURE_AI_FOUNDRY_ENDPOINT")
MODEL_DEPLOYMENT_NAME = os.getenv("AZURE_OPENAI_DEPLOYMENT_NAME")
PROJECT_NAME = os.getenv("AZURE_AI_PROJECT_NAME")
APP_INSIGHTS_CONNECTION_STRING = os.getenv("APPLICATIONINSIGHTS_CONNECTION_STRING", "")
CH05_AGENT_ID = os.getenv("CH05_AGENT_ID", "")

print("Loaded environment values (masked):")
print(f"PROJECT_ENDPOINT set: {bool(PROJECT_ENDPOINT)}")
print(f"MODEL_DEPLOYMENT_NAME set: {bool(MODEL_DEPLOYMENT_NAME)}")
print(f"PROJECT_NAME set: {bool(PROJECT_NAME)}")
print(f"APP_INSIGHTS_CONNECTION_STRING set: {bool(APP_INSIGHTS_CONNECTION_STRING)}")
print(f"CH05_AGENT_ID set: {bool(CH05_AGENT_ID)}")

## Section 1 - Environment Readiness Task

### Student Task 1
Fill in the assertions so the notebook fails fast when required settings are missing.

Include your previously deployed agent ID in `CH05_AGENT_ID` (from Challenge 03 or Challenge 04).

Hint:
- You need endpoint and model deployment for all later sections.
- `CH05_AGENT_ID` should point to an existing agent, not a new one created for this challenge.

In [ ]:
assert ___, "AZURE_AI_FOUNDRY_ENDPOINT is missing in .env"
assert ___, "AZURE_OPENAI_DEPLOYMENT_NAME is missing in .env"
assert ___, "CH05_AGENT_ID is missing in .env (set this to an agent from earlier challenges)"
print("Readiness checks passed.")

In [ ]:
from azure.identity import AzureCliCredential, ChainedTokenCredential, InteractiveBrowserCredential

tenant_id = os.getenv("AZURE_TENANT_ID")
credential = ChainedTokenCredential(
    AzureCliCredential(tenant_id=tenant_id) if tenant_id else AzureCliCredential(),
    InteractiveBrowserCredential(tenant_id=tenant_id) if tenant_id else InteractiveBrowserCredential(),
)

credential.get_token("https://management.azure.com/.default")
print("Authenticated to Azure.")


In [ ]:
# Student Task 1.5
from azure.ai.projects import AIProjectClient

project_client = AIProjectClient(endpoint=___, credential=credential)
agents_client = project_client.agents

thread = agents_client.threads.create()
prompt = ___  # Write an observability-focused prompt for your existing agent

agents_client.messages.create(thread_id=thread.id, role="user", content=prompt)
run = agents_client.runs.create_and_process(thread_id=thread.id, agent_id=___)

messages = agents_client.messages.list(thread_id=thread.id)
assistant_preview = ""
for m in messages:
    if getattr(m, "role", "") == "assistant" and getattr(m, "content", None):
        try:
            assistant_preview = m.content[0].text.value
        except Exception:
            assistant_preview = str(m.content)
        break

real_run_summary = {
    "agent_id": CH05_AGENT_ID,
    "thread_id": thread.id,
    "run_id": run.id,
    "prompt": prompt,
    "assistant_preview": assistant_preview[:200],
}

print(json.dumps(real_run_summary, indent=2))

## Section 1.5 - Real Run on an Existing Agent (Required)

### Student Task 1.5
Run one real interaction against the existing agent in `CH05_AGENT_ID` using the SDK.

You must capture evidence fields for observability correlation:
- `agent_id`
- `thread_id`
- `run_id`
- `prompt`
- `assistant_preview`

Hint:
- Reuse the same `credential` object from the previous cell.
- Use `AIProjectClient(...).agents` and `runs.create_and_process(...)`.

## Section 2 - Tracing and Traceability

This section mirrors the observability demo idea: create traces for each turn, then inspect span-level attributes.

You will first run a local tracing simulation so you can see exactly what telemetry is captured.


In [ ]:
from opentelemetry import trace
from opentelemetry.sdk.resources import Resource
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import ConsoleSpanExporter, SimpleSpanProcessor
from opentelemetry.sdk.trace.export.in_memory_span_exporter import InMemorySpanExporter

resource = Resource.create({
    "service.name": "wth-foundry-observability",
    "challenge": "05",
})
provider = TracerProvider(resource=resource)
memory_exporter = InMemorySpanExporter()
provider.add_span_processor(SimpleSpanProcessor(memory_exporter))
provider.add_span_processor(SimpleSpanProcessor(ConsoleSpanExporter()))
trace.set_tracer_provider(provider)
tracer = trace.get_tracer(__name__)

print("Tracer initialized.")


### Student Task 2 - Instrument a Turn

Complete the tracing attributes in the function below.

Hint:
- Use meaningful values for prompt length, response length, and simulated token count.
- Keep attribute names stable so dashboards can aggregate them later.


In [ ]:
def traced_turn(user_prompt: str) -> str:
    start = time.perf_counter()
    with tracer.start_as_current_span("student.agent.turn") as span:
        span.set_attribute("genai.operation.name", "chat_turn")
        span.set_attribute("genai.prompt.length", ___)

        # Simulate model work
        response = f"Answer generated for: {user_prompt}"
        time.sleep(0.15)

        span.set_attribute("genai.response.length", ___)
        span.set_attribute("genai.usage.total_tokens", ___)
        elapsed_ms = (time.perf_counter() - start) * 1000
        span.set_attribute("genai.latency.ms", round(elapsed_ms, 2))

        return response


In [ ]:
prompts = [
    "Plan a 3-day trip to Lisbon with budget priorities.",
    "Suggest two safer alternatives if one attraction is closed.",
    "Summarize the itinerary in bullet points.",
]

rows = []
for p in prompts:
    answer = traced_turn(p)
    rows.append({"prompt": p, "response_preview": answer[:60] + "..."})

pd.DataFrame(rows)


In [ ]:
finished_spans = memory_exporter.get_finished_spans()
print(f"Captured spans: {len(finished_spans)}")

span_rows = []
for s in finished_spans:
    attrs = dict(s.attributes)
    span_rows.append({
        "span_name": s.name,
        "prompt_len": attrs.get("genai.prompt.length"),
        "response_len": attrs.get("genai.response.length"),
        "tokens": attrs.get("genai.usage.total_tokens"),
        "latency_ms": attrs.get("genai.latency.ms"),
    })

pd.DataFrame(span_rows)


## Section 3 - Monitoring Signals

In the real Foundry monitoring dashboard, you inspect token usage, latency, and success rate over time.

### Student Task 3
Compute p95 latency and success rate from the synthetic run log below.

Hint:
- p95 is the 95th percentile.
- Success rate is successful runs divided by total runs.


In [ ]:
run_log = pd.DataFrame([
    {"run_id": "r1", "latency_ms": 900, "tokens": 340, "status": "success"},
    {"run_id": "r2", "latency_ms": 1250, "tokens": 410, "status": "success"},
    {"run_id": "r3", "latency_ms": 870, "tokens": 300, "status": "failed"},
    {"run_id": "r4", "latency_ms": 1600, "tokens": 520, "status": "success"},
    {"run_id": "r5", "latency_ms": 1100, "tokens": 390, "status": "success"},
])

p95_latency = ___
success_rate = ___

print(f"p95 latency: {p95_latency:.2f} ms")
print(f"success rate: {success_rate:.2%}")


## Section 4 - Continuous Evaluation

In Foundry, continuous evaluation runs on live traffic with configured evaluators.

For this lab, you will build a small local evaluator function first so you understand the idea before configuring platform rules.


In [ ]:
def simple_task_adherence(prompt: str, response: str) -> dict:
    """Small teaching evaluator: checks if key prompt terms appear in response."""
    key_terms = [w.lower() for w in prompt.split() if len(w) > 4]
    matched = sum(1 for t in key_terms if t in response.lower())

    # Student Task 4: fill in score and pass/fail logic
    score = ___  # Example: matched / max(len(key_terms), 1)
    passed = ___  # Example: score >= 0.4

    return {
        "evaluator": "task_adherence_local",
        "score": round(score, 3),
        "passed": bool(passed),
        "matched_terms": matched,
        "total_terms": len(key_terms),
    }


In [ ]:
example_prompt = "Create an itinerary with budget and safety guidance"
example_response = "Here is an itinerary with budget suggestions and safety guidance for each day."

simple_task_adherence(example_prompt, example_response)


## Section 5 - Batch Evaluation

The demo repository includes a batch evaluation step on a JSONL dataset.

### Student Task 5
Complete at least two test queries, then run the local batch loop.

Hint:
- Write prompts that test different behaviors: planning quality, coherence, and safety awareness.


In [ ]:
batch_queries = [
    {"id": "q1", "prompt": "Plan a family-friendly weekend in Barcelona"},
    {"id": "q2", "prompt": ___},
    {"id": "q3", "prompt": ___},
]

with open("test-queries-ch05.jsonl", "w", encoding="utf-8") as f:
    for item in batch_queries:
        f.write(json.dumps(item) + "\n")

print("Wrote test-queries-ch05.jsonl")


In [ ]:
results = []
for item in batch_queries:
    simulated_response = f"Draft answer for: {item['prompt']}"
    eval_result = simple_task_adherence(item["prompt"], simulated_response)
    results.append({
        "id": item["id"],
        "prompt": item["prompt"],
        "score": eval_result["score"],
        "passed": eval_result["passed"],
    })

pd.DataFrame(results)


## Section 6 - Connect to Foundry Monitoring and Evaluation

Run these steps in the portal after completing the notebook coding tasks:

- Open your project in Foundry.
- Use the existing agent referenced by `CH05_AGENT_ID` (created in earlier challenges).
- Go to Agents -> Traces and verify spans are arriving for that agent.
- Go to Agent Monitor and inspect latency, usage, and success charts.
- Configure a continuous evaluation rule for that same existing agent and generate traffic.
- Compare your local learning from this notebook with platform-level metrics.

Do not create a new throwaway agent for this challenge unless your coach asks you to.

## Reflection (Required)

Write short answers:

1. Which metric would you alert on first in production, and why?
2. What did traces tell you that plain logs would not?
3. Which type of evaluation (continuous or batch) gave you the most useful signal for this agent?
4. Name one improvement you would implement before releasing this agent to users.


In [ ]:
# Student Task 6: Write your reflection as a Python dict and run this cell.
reflection = {
    "metric_alert": "",
    "trace_vs_logs": "",
    "best_eval_signal": "",
    "pre_release_improvement": "",
}

assert all(v.strip() for v in reflection.values()), "Please complete all reflection fields."
print("Reflection completed. Great work on Challenge 05.")
